# Sesión 11: Clustering y Segmentación

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Agrupar datos sin etiquetas usando algoritmos de clustering, y evaluar objetivamente la calidad de los grupos obtenidos.

## 🗺️ Tabla de Contenido
1. [Introducción: el mundo sin etiquetas](#intro)
2. [K-Means con datos sintéticos (make_blobs)](#kmeans-blobs)
3. [Método del Codo](#codo)
4. [Dataset real: Mall Customer Segmentation](#mall)
5. [Clustering Jerárquico](#jerarquico)
6. [DBSCAN](#dbscan)
7. [Métricas internas de clustering](#metricas)
8. [Perfilado de clusters](#perfilado)
9. [Ejemplos de aplicación real](#aplicaciones)
10. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

Hasta ahora, todos nuestros modelos tenían una "respuesta correcta" (`y`) para aprender. En **clustering** no hay etiquetas: el algoritmo debe descubrir por sí mismo qué puntos se parecen entre sí y agruparlos. Es la herramienta central de la **segmentación de clientes**.

<a id="kmeans-blobs"></a>
## 2. K-Means con Datos Sintéticos (make_blobs)

### 🔬 Teoría técnica
K-Means agrupa los datos en `K` clusters iterando dos pasos hasta converger:
1. Asigna cada punto al centroide más cercano.
2. Recalcula cada centroide como el promedio de los puntos asignados a él.

**Hiperparámetros clave:** `n_clusters` (K), `init` (`k-means++` inicializa los centroides de forma inteligente, evitando malos arranques), `max_iter`.

Usamos `make_blobs` primero porque, al conocer de antemano los centros reales, podemos **ver claramente** si el algoritmo los encuentra bien.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import numpy as np

X_sinteticos, y_real = make_blobs(
    n_samples=300, centers=4, cluster_std=0.9, random_state=42
)

kmeans = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=42)
etiquetas = kmeans.fit_predict(X_sinteticos)

fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(X_sinteticos[:, 0], X_sinteticos[:, 1], c=etiquetas, cmap="viridis", alpha=0.7)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", marker="X", s=200, label="Centroides")
ax.set_title("K-Means sobre datos sintéticos (make_blobs)")
ax.legend()
plt.show()

### 🧠 Resumen para dummies
K-Means "adivina" un centro por grupo, mueve el centro al promedio real de sus puntos, reasigna, y repite hasta que ya no cambia — como reubicar sucursales de una tienda hasta que quedan justo en el centro de cada zona de clientes.

<a id="codo"></a>
## 3. Método del Codo (Elbow Method)

### 🔬 Teoría técnica
¿Cómo elegir K? Se calcula la **inercia** (suma de distancias al cuadrado de cada punto a su centroide) para varios valores de K. La inercia siempre baja al aumentar K, pero llega un punto donde bajar más ya no compensa la complejidad extra — ese "codo" es el K sugerido.

In [ ]:
inercias = []
rango_k = range(1, 9)

for k in rango_k:
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    modelo.fit(X_sinteticos)
    inercias.append(modelo.inertia_)

plt.plot(rango_k, inercias, marker="o")
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inercia")
plt.title("Método del Codo")
plt.show()

### 🧠 Resumen para dummies
Busca el punto donde la curva deja de bajar "en picada" y empieza a aplanarse — ese es tu K candidato (aquí debería notarse claramente en K=4, que es como se generaron los datos).

<a id="mall"></a>
## 4. Dataset Real: Mall Customer Segmentation

Ahora aplicamos lo aprendido a un caso real: segmentar clientes de un centro comercial según su ingreso anual y su "puntaje de gasto".

In [ ]:
import pandas as pd

try:
    import kagglehub
    import os
    path = kagglehub.dataset_download("vjchoudhary7/customer-segmentation-tutorial-in-python")
    df = pd.read_csv(os.path.join(path, "Mall_Customers.csv"))
except Exception as e:
    print("No se pudo usar Kaggle, usando fuente remota alternativa:", e)
    url = (
        "https://raw.githubusercontent.com/SteffiPeTaffy/machineLearningAZ/master/"
        "Machine%20Learning%20A-Z%20Template%20Folder/Part%204%20-%20Clustering/"
        "Section%2025%20-%20Hierarchical%20Clustering/Mall_Customers.csv"
    )
    df = pd.read_csv(url)

df.columns = [c.strip().replace(" ", "_").replace("(", "").replace(")", "").replace("$", "").lower() for c in df.columns]
df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Escalamos SIEMPRE antes de clusterizar con K-Means/DBSCAN (son sensibles a la escala)
X_mall = df[["annual_income_k", "spending_score_1-100"]] if "annual_income_k" in df.columns else df.filter(regex="income|spending")
X_mall.columns = ["ingreso_anual", "puntaje_gasto"]

escalador = StandardScaler()
X_mall_esc = escalador.fit_transform(X_mall)

# Método del codo sobre los datos reales
inercias_mall = []
for k in range(1, 9):
    modelo = KMeans(n_clusters=k, n_init=10, random_state=42)
    modelo.fit(X_mall_esc)
    inercias_mall.append(modelo.inertia_)

plt.plot(range(1, 9), inercias_mall, marker="o")
plt.xlabel("K")
plt.ylabel("Inercia")
plt.title("Método del Codo — Mall Customers")
plt.show()

In [ ]:
kmeans_mall = KMeans(n_clusters=5, n_init=10, random_state=42)
df["cluster"] = kmeans_mall.fit_predict(X_mall_esc)

fig, ax = plt.subplots(figsize=(6, 5))
sns_scatter = ax.scatter(X_mall["ingreso_anual"], X_mall["puntaje_gasto"], c=df["cluster"], cmap="tab10")
ax.set_xlabel("Ingreso Anual (miles $)")
ax.set_ylabel("Puntaje de Gasto (1-100)")
ax.set_title("Segmentación de Clientes del Centro Comercial")
plt.show()

<a id="jerarquico"></a>
## 5. Clustering Jerárquico

### 🔬 Teoría técnica
Construye una jerarquía de clusters (visualizable como un **dendrograma**), uniendo progresivamente los puntos/clusters más cercanos. **Hiperparámetros clave:** `n_clusters`, `linkage` (`ward`, `average`, `complete`).

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

# Dendrograma sobre una muestra para que sea legible
muestra = X_mall_esc[:40]
enlaces = linkage(muestra, method="ward")

plt.figure(figsize=(10, 4))
dendrogram(enlaces)
plt.title("Dendrograma (muestra de 40 clientes)")
plt.xlabel("Cliente")
plt.ylabel("Distancia")
plt.show()

jerarquico = AgglomerativeClustering(n_clusters=5, linkage="ward")
clusters_jerarquico = jerarquico.fit_predict(X_mall_esc)

### 🧠 Resumen para dummies
El dendrograma es como un "árbol genealógico" de los datos: mientras más abajo se unen dos puntos, más parecidos son. Cortar el árbol a cierta altura te da el número de clusters.

<a id="dbscan"></a>
## 6. DBSCAN

### 🔬 Teoría técnica
Agrupa puntos según su **densidad**: forma clusters donde hay muchos puntos cercanos entre sí, y marca como "ruido" (`-1`) los puntos aislados. No requiere definir K de antemano. **Hiperparámetros clave:** `eps` (distancia máxima entre vecinos), `min_samples`.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
clusters_dbscan = dbscan.fit_predict(X_mall_esc)

n_clusters_dbscan = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
n_ruido = list(clusters_dbscan).count(-1)
print(f"DBSCAN encontró {n_clusters_dbscan} clusters y {n_ruido} puntos de ruido")

### 💪 Fortalezas y debilidades

| Algoritmo | Fortaleza | Debilidad |
|---|---|---|
| K-Means | Rápido, simple, escala bien | Requiere K de antemano, asume clusters esféricos |
| Jerárquico | No requiere K de antemano, dendrograma interpretable | Costoso en datasets grandes |
| DBSCAN | Detecta formas arbitrarias y ruido/outliers | Sensible a `eps`, dificultad con densidades muy distintas |

<a id="metricas"></a>
## 7. Métricas Internas de Clustering

### 🔬 Teoría técnica
- **Silhouette Score** (rango [-1, 1], más alto mejor): compara qué tan cerca está un punto de su propio cluster vs. del cluster más cercano.
- **Davies-Bouldin** (más bajo mejor): relación entre dispersión intra-cluster y distancia inter-cluster.
- **Calinski-Harabasz** (más alto mejor): dispersión entre clusters vs. dentro de los clusters.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

resultados = []
for nombre, etiquetas_modelo in [
    ("K-Means", df["cluster"]),
    ("Jerárquico", clusters_jerarquico),
    ("DBSCAN", clusters_dbscan),
]:
    if len(set(etiquetas_modelo)) > 1:
        resultados.append({
            "modelo": nombre,
            "silhouette": silhouette_score(X_mall_esc, etiquetas_modelo),
            "davies_bouldin": davies_bouldin_score(X_mall_esc, etiquetas_modelo),
            "calinski_harabasz": calinski_harabasz_score(X_mall_esc, etiquetas_modelo),
        })

pd.DataFrame(resultados)

<a id="perfilado"></a>
## 8. Perfilado de Clusters

### 🔬 Teoría técnica
Un cluster sin interpretación de negocio no sirve de nada. El último paso siempre es describir cada grupo con palabras: "¿quiénes son estos clientes?".

In [ ]:
perfil = pd.concat([X_mall, df["cluster"]], axis=1).groupby("cluster").mean()
perfil

### 🧠 Resumen para dummies
Con el promedio de ingreso y gasto por cluster puedes ponerle nombre a cada grupo, ej.: "ingreso alto - gasto alto" (clientes premium), "ingreso bajo - gasto alto" (compradores impulsivos), etc.

<a id="aplicaciones"></a>
## 9. Ejemplos de Aplicación en el Mundo Real

- Segmentar clientes para campañas de marketing personalizadas (el caso de hoy).
- Agrupar documentos o noticias por temática sin etiquetas previas.
- DBSCAN para detectar anomalías: los puntos marcados como "ruido" pueden ser fraudes o errores de datos.

<a id="retos"></a>
## 10. Retos de Práctica

### 🥉 Reto Básico
Sobre `make_blobs` con `centers=3`, aplica K-Means con K=3 y grafica los clusters junto a sus centroides.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Sobre `Mall_Customers`, usa el Método del Codo para confirmar/ajustar el K elegido (K=5 en el ejemplo), y calcula el Silhouette Score para justificar tu elección final de K.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Compara K-Means, Clustering Jerárquico y DBSCAN sobre `Mall_Customers` incluyendo también la variable `Age` (edad) además de ingreso y gasto (3 variables). Genera la tabla de métricas internas y un perfil descriptivo (promedios) de cada cluster resultante del mejor modelo según Silhouette Score.

In [ ]:
# Tu solución al Reto Avanzado aquí
